# 01 — LangChain & LangGraph Demo (In-Memory Object Flows)

Interactive demonstration of Didactiq's **context formatters**, **LangChain prompt chains**, **JSON response parser**, and **LangGraph `StateGraph` + `MemorySaver` + `InMemoryStore`** using pure in-memory objects (no database required).

- Set `USE_REAL_LLM = False` (default) for instant deterministic responses.
- Set `USE_REAL_LLM = True` (with `OPENROUTER_API_KEY` set in `backend/.env`) to watch live OpenRouter outputs.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "backend"):
    if (candidate / "src" / "main.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from notebooks._common import (
    banner,
    get_notebook_llm,
    pretty_print,
    render_visualization,
    setup_backend_env,
)

setup_backend_env()

USE_REAL_LLM = False
llm = get_notebook_llm(use_real_llm=USE_REAL_LLM)
print(f"Active LLM: {type(llm).__name__} (USE_REAL_LLM={USE_REAL_LLM})")

## 1. In-Memory `ChatPayload` -> Prompt Context Formatters

In [ ]:
from src.lib.formatter import (
    format_analysis_config_context,
    format_classroom_context,
    format_students_context,
)
from src.types.schemas import (
    AnalysisConfig,
    ChatPayload,
    Instruction,
    Material,
    Message,
    Student,
    StudentCustomField,
    StudentGrade,
)

payload = ChatPayload(
    messages=[
        Message(id="m1", role="user", text="We finished Kinetics Lab 1.", timestamp="2026-11-11T09:00:00Z"),
        Message(id="m2", role="assistant", text="Ready to analyze!", timestamp="2026-11-11T09:00:05Z"),
        Message(id="m3", role="user", text="Which concepts need review?", timestamp="2026-11-11T09:01:00Z"),
    ],
    workspaceName="Fall 2026 Science",
    classItemName="AP Chemistry 101",
    subject="AP Chemistry",
    academicYear="2026-2027",
    semester="Fall",
    teacherName="Dr. Marie Curie",
    teachingStyle="Socratic Method",
    materials=[Material(name="Chemical Kinetics Guide", type="Study Material", tags=["kinetics", "rate-laws"])],
    instructions=[Instruction(title="Feedback Tone", type="Grading Rubric", content="Be concise and constructive.")],
    students=[
        Student(
            name="Ada Lovelace",
            rollNumber="CHEM-01",
            email="ada@school.edu",
            performanceIndicator="High",
            attendance=98.0,
            grades=[StudentGrade(assessmentName="Lab 1", score=94.0, maxScore=100.0)],
            customFields=[StudentCustomField(label="Track", value="Honors")],
        ),
        Student(
            name="Alan Turing",
            rollNumber="CHEM-02",
            email="alan@school.edu",
            performanceIndicator="Needs Review",
            attendance=91.0,
            grades=[StudentGrade(assessmentName="Lab 1", score=70.0, maxScore=100.0)],
        ),
    ],
    newAnalysisConfig=AnalysisConfig(
        type="diagnostic",
        scopeType="class",
        customInstructions="Highlight prerequisite gaps.",
    ),
)

class_ctx = format_classroom_context(payload)
students_ctx = format_students_context(payload.students)
config_ctx = format_analysis_config_context(payload.newAnalysisConfig)

pretty_print("Formatted Class Context", class_ctx)
pretty_print("Formatted Students Context", students_ctx)
pretty_print("Formatted Config Context", config_ctx)

## 2. LangChain `ChatPromptTemplate | LLM` + `parse_llm_response`

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate

from src.lib.llm import parse_llm_response

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(
        content=(
            "You are an AI pedagogical assistant. "
            'Return strictly JSON: {"text": "markdown summary", "visualization": null}.'
        )
    ),
    ("placeholder", "{history}"),
    (
        "user",
        "CLASS CONTEXT:\n{class_ctx}\n\nSTUDENTS:\n{students_ctx}\n\nCONFIG:\n{config_ctx}\n\nQUESTION: {question}",
    ),
])

chain = prompt | llm
history = [(m.role, m.text) for m in payload.messages[:-1]]

ai_msg = await chain.ainvoke({
    "history": history,
    "class_ctx": class_ctx,
    "students_ctx": students_ctx,
    "config_ctx": config_ctx,
    "question": payload.messages[-1].text,
})

banner("DEMO 2", "LangChain ChatPromptTemplate | LLM + parse_llm_response")
parsed = parse_llm_response(ai_msg.content)
pretty_print("Parsed Chain Output", parsed)
render_visualization(parsed.get("visualization") if isinstance(parsed.get("visualization"), dict) else None)

## 3. LangGraph `StateGraph` + `MemorySaver` + `InMemoryStore` (Zero Database)

Demonstrates conditional soft-warning routing, multi-turn thread checkpointing in `MemorySaver`, and in-memory teacher preferences in `InMemoryStore`.

In [ ]:
import json
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.store.memory import InMemoryStore


class CopilotState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    coverage_score: float
    confirmed: bool
    soft_warning_issued: bool
    reply_json: dict[str, object] | None


class CopilotInput(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]
    coverage_score: float
    confirmed: bool
    soft_warning_issued: bool
    reply_json: dict[str, object] | None


memory = MemorySaver()
store = InMemoryStore()
store.put(("teacher", "dr-curie"), "preferences", {"tone": "Socratic", "max_bullets": 3})


async def gate_node(state: CopilotState) -> dict[str, object]:
    return {"soft_warning_issued": state["coverage_score"] < 0.4 and not state["confirmed"]}


def route_gate(state: CopilotState) -> str:
    return "soft_warning" if state["soft_warning_issued"] else "diagnose"


async def soft_warning_node(state: CopilotState) -> dict[str, object]:
    warning = {
        "text": f"Coverage is {round(state['coverage_score'] * 100)}%. Confirm to proceed anyway.",
        "requires_confirmation": True,
    }
    return {"messages": [AIMessage(content=json.dumps(warning))], "reply_json": warning}


async def diagnose_node(state: CopilotState) -> dict[str, object]:
    prefs = store.get(("teacher", "dr-curie"), "preferences")
    tone = prefs.value["tone"] if prefs else "Standard"
    prompt_msgs = [
        SystemMessage(content=f"You are an AI pedagogical assistant. Tone: {tone}. Return JSON with 'text'."),
        *state["messages"],
    ]
    msg = await llm.ainvoke(prompt_msgs)
    parsed_reply = parse_llm_response(msg.content)
    return {"messages": [AIMessage(content=json.dumps(parsed_reply))], "reply_json": parsed_reply}


builder = StateGraph(CopilotState, input_schema=CopilotInput)
builder.add_node("gate", gate_node)
builder.add_node("soft_warning", soft_warning_node)
builder.add_node("diagnose", diagnose_node)
builder.add_edge(START, "gate")
builder.add_conditional_edges("gate", route_gate, {"soft_warning": "soft_warning", "diagnose": "diagnose"})
builder.add_edge("soft_warning", END)
builder.add_edge("diagnose", END)
graph = builder.compile(checkpointer=memory)

config: RunnableConfig = {"configurable": {"thread_id": "manual-session-1"}}

# Turn 1: low coverage -> soft warning
turn1 = await graph.ainvoke(
    {
        "messages": [HumanMessage(content="Analyze Chemical Kinetics readiness")],
        "coverage_score": 0.25,
        "confirmed": False,
        "soft_warning_issued": False,
        "reply_json": None,
    },
    config=config,
)
banner("DEMO 3", "LangGraph Multi-Turn Checkpointer + InMemoryStore")
pretty_print("Turn 1 (Soft Warning)", turn1["reply_json"])

# Turn 2: confirm continuation on same thread_id -> runs LLM diagnosis
turn2 = await graph.ainvoke(
    {"messages": [HumanMessage(content="Yes, continue anyway.")], "confirmed": True},
    config=config,
)
pretty_print("Turn 2 (Confirmed Diagnosis)", turn2["reply_json"])
print("Checkpointed message count in memory:", len(graph.get_state(config).values["messages"]))